In [1]:
import os, zipfile
from pathlib import Path
import pandas as pd

RAW = Path("data/raw")
zip_path = RAW / "FinSurvival2025_Dataset.zip"
assert zip_path.exists(), f"ZIP not found at {zip_path}"

extract_dir = RAW / "finsurvival"
if not extract_dir.exists():
    print(f"unzipping to {extract_dir}...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    print("done")
else:
    print(f"already extracted at {extract_dir}")

# Inventory
for root, dirs, files in os.walk(extract_dir):
    rel = Path(root).relative_to(RAW)
    for f in files:
        p = Path(root) / f
        size_mb = p.stat().st_size / 1e6
        print(f"  {rel/f}  —  {size_mb:,.1f} MB")

unzipping to data\raw\finsurvival...
done
  finsurvival\participant_data.tar.gz  —  8,931.6 MB
  finsurvival\test_features.tar.gz  —  2,741.6 MB
  finsurvival\__MACOSX\._participant_data.tar.gz  —  0.0 MB
  finsurvival\__MACOSX\._test_features.tar.gz  —  0.0 MB


In [8]:
DATA_ROOT = Path("data/raw/finsurvival/participant_data/participant_data")
TEST_ROOT = Path("data/raw/finsurvival/test_features")

INDEX_EVENTS   = ["Borrow", "Deposit", "Repay", "Withdraw"]
OUTCOME_EVENTS = ["Borrow", "Deposit", "Repay", "Withdraw", "Liquidated"]

TASKS = []
for ie in INDEX_EVENTS:
    for oe in OUTCOME_EVENTS:
        if ie == oe:
            continue
        task_dir = DATA_ROOT / ie / oe
        if task_dir.exists():
            TASKS.append({"ie": ie, "oe": oe, "task_id": f"{ie}_{oe}", "dir": task_dir})

print(f"found {len(TASKS)} tasks (expected 16)\n")
total_train, total_test = 0, 0
for t in TASKS:
    tr = (t["dir"] / "train.csv").stat().st_size / 1e6
    te = (t["dir"] / "test_features.csv").stat().st_size / 1e6
    total_train += tr; total_test += te
    print(f"  {t['task_id']:25s}  train {tr:>7,.0f} MB  test {te:>6,.0f} MB")
print(f"\n  TOTAL: train {total_train/1024:.1f} GB, test {total_test/1024:.1f} GB")

found 16 tasks (expected 16)

  Borrow_Deposit             train   1,078 MB  test    204 MB
  Borrow_Repay               train   1,029 MB  test    178 MB
  Borrow_Withdraw            train   1,081 MB  test    205 MB
  Borrow_Liquidated          train   1,093 MB  test    206 MB
  Deposit_Borrow             train   2,541 MB  test    529 MB
  Deposit_Repay              train   2,536 MB  test    528 MB
  Deposit_Withdraw           train   2,315 MB  test    406 MB
  Deposit_Liquidated         train   2,572 MB  test    535 MB
  Repay_Borrow               train     924 MB  test    187 MB
  Repay_Deposit              train     971 MB  test    213 MB
  Repay_Withdraw             train     977 MB  test    214 MB
  Repay_Liquidated           train     987 MB  test    216 MB
  Withdraw_Borrow            train   1,842 MB  test    725 MB
  Withdraw_Deposit           train   1,614 MB  test    602 MB
  Withdraw_Repay             train   1,842 MB  test    725 MB
  Withdraw_Liquidated        train   1,8

In [9]:
task = TASKS[0]  # Borrow_Deposit
print(f"=== loading sample from {task['task_id']} ===")
sample = pd.read_csv(task["dir"] / "train.csv", nrows=5000)
print(f"shape: {sample.shape}")

print(f"\n=== all {len(sample.columns)} columns ===")
for i, c in enumerate(sample.columns):
    print(f"  [{i:3d}] {c:50s} dtype: {str(sample[c].dtype):10s} nunique: {sample[c].nunique()}")

print(f"\n=== target columns ===")
print(f"  timeDiff:  min={sample['timeDiff'].min()}, max={sample['timeDiff'].max()}, "
      f"median={sample['timeDiff'].median():.1f}, mean={sample['timeDiff'].mean():.1f}")
print(f"  status:    {sample['status'].value_counts().to_dict()}  (1=event occurred, 0=censored)")
print(f"  Index Event:   {sample['Index Event'].unique()}")
print(f"  Outcome Event: {sample['Outcome Event'].unique()}")

=== loading sample from Borrow_Deposit ===
shape: (5000, 92)

=== all 92 columns ===
  [  0] userBorrowSum                                      dtype: float64    nunique: 3450
  [  1] marketWithdrawAvgAmount                            dtype: float64    nunique: 2396
  [  2] marketDepositAvgAmountUSD                          dtype: float64    nunique: 3384
  [  3] sinDayOfMonth                                      dtype: float64    nunique: 23
  [  4] userSecondsSinceFirstTransaction                   dtype: float64    nunique: 3826
  [  5] timeOfDay                                          dtype: float64    nunique: 4521
  [  6] userActiveDaysWeekly                               dtype: float64    nunique: 7
  [  7] userLiquidationCount                               dtype: int64      nunique: 2
  [  8] userActiveDaysMonthly                              dtype: float64    nunique: 21
  [  9] userRepayCount                                     dtype: int64      nunique: 806
  [ 10] cosQuart

In [10]:
# Identify feature categories (90 features expected per FinSurvival spec)
NON_FEATURE = {"timeDiff", "status", "Index Event", "Outcome Event", "user", "pool", "reserve",
               "coinType", "userCoinTypeMode", "userReserveMode", "type", "timestamp"}

feature_cols = [c for c in sample.columns if c not in NON_FEATURE]
user_feats = [c for c in feature_cols if c.lower().startswith("user")]
mkt_feats  = [c for c in feature_cols if c.lower().startswith("market")]
time_feats = [c for c in feature_cols if any(k in c.lower() for k in
    ["day", "time", "quarter", "weekend", "cos", "sin", "year", "month"])]
tx_feats   = [c for c in feature_cols if c not in user_feats + mkt_feats + time_feats]

print(f"total features:        {len(feature_cols)}")
print(f"  user-centric:        {len(user_feats)}")
print(f"  market-level:        {len(mkt_feats)}")
print(f"  temporal/cyclical:   {len(time_feats)}")
print(f"  transactional/other: {len(tx_feats)}  → {tx_feats}")

total features:        80
  user-centric:        31
  market-level:        25
  temporal/cyclical:   24
  transactional/other: 5  → ['priceInUSD', 'amount', 'amountUSD', 'logAmount', 'logAmountUSD']


In [11]:
# Verify timestamp ranges match FinSurvival spec
# TRAIN: 2022-03-12 to 2024-07-02 | TEST: 2024-07-03 to 2025-08-21
print(f"=== chronological check ({task['task_id']}) ===")

sample_full = pd.read_csv(task["dir"] / "train.csv", usecols=["timestamp"])
ts_min = pd.to_datetime(sample_full["timestamp"].min(), unit='s')
ts_max = pd.to_datetime(sample_full["timestamp"].max(), unit='s')
print(f"train.csv timestamp range: {ts_min}  →  {ts_max}  ({len(sample_full):,} rows)")

sample_test = pd.read_csv(task["dir"] / "test_features.csv", usecols=["timestamp"])
tst_min = pd.to_datetime(sample_test["timestamp"].min(), unit='s')
tst_max = pd.to_datetime(sample_test["timestamp"].max(), unit='s')
print(f"test_features.csv range:   {tst_min}  →  {tst_max}  ({len(sample_test):,} rows)")

print(f"\n→ expected train end: 2024-07-02 | expected test start: 2024-07-03")
assert ts_max < tst_min, "LEAKAGE: train overlaps test chronologically!"
print("✓ no chronological overlap between train and test")

=== chronological check (Borrow_Deposit) ===
train.csv timestamp range: 2022-03-14 12:40:09  →  2024-07-02 15:27:36  (882,736 rows)
test_features.csv range:   2024-08-01 15:29:12  →  2025-07-21 21:52:49  (164,631 rows)

→ expected train end: 2024-07-02 | expected test start: 2024-07-03
✓ no chronological overlap between train and test


In [12]:
import json

PROC = Path("data/processed")
PROC.mkdir(parents=True, exist_ok=True)

# Task registry (16 entries)
task_registry = [
    {"task_id": t["task_id"], "index_event": t["ie"], "outcome_event": t["oe"],
     "train_path": str((t["dir"] / "train.csv").relative_to(Path("data/raw"))),
     "test_path":  str((t["dir"] / "test_features.csv").relative_to(Path("data/raw")))}
    for t in TASKS
]
with open(PROC / "task_registry.json", "w") as f:
    json.dump(task_registry, f, indent=2)
print(f"✓ saved {len(task_registry)} tasks → {PROC/'task_registry.json'}")

# Feature inventory
feat_dict = {
    "n_features": len(feature_cols),
    "features":   feature_cols,
    "user_feats": user_feats,
    "mkt_feats":  mkt_feats,
    "time_feats": time_feats,
    "tx_feats":   tx_feats,
    "targets":    ["timeDiff", "status"],
    "metadata":   ["user", "pool", "reserve", "coinType", "type", "timestamp",
                   "Index Event", "Outcome Event"],
}
with open(PROC / "feature_list.json", "w") as f:
    json.dump(feat_dict, f, indent=2)
print(f"✓ saved feature list → {PROC/'feature_list.json'}")

print(f"\nPhase 1 complete.")
print(f"  16 tasks registered")
print(f"  {len(feature_cols)} features identified")
print(f"  ~{total_train/1024:.1f} GB train + ~{total_test/1024:.1f} GB test ready for Phase 2")

✓ saved 16 tasks → data\processed\task_registry.json
✓ saved feature list → data\processed\feature_list.json

Phase 1 complete.
  16 tasks registered
  80 features identified
  ~24.7 GB train + ~6.3 GB test ready for Phase 2
